# Reasoning Memory: Remember WHY You Decided, Not Just What You Know

The earlier demos store what the agent knows. This demo stores **why it decided**: the question → tool steps → evidence → outcome chain, with provenance. Without it, "why did you recommend X?" gets a confabulated answer: the agent invents a plausible justification because the real reasoning was never kept.

> **Honesty note:** "reasoning memory" is an **engineering pattern**, not an established category in academic memory taxonomies. What the research does support is the value of traceability and provenance in agent memory: MemWeaver ([arXiv 2601.18204](https://arxiv.org/abs/2601.18204)) and the Engram system ([arXiv 2606.09900](https://arxiv.org/abs/2606.09900), single-author preprint).

Two tracks, same traces:
- **Key-value** (`agent.state`): a Strands `HookProvider` records each invocation's trace automatically (zero changes to the tools).
- **Graph** (Neo4j): the same traces as node chains with provenance edges, enabling the **reverse audit** a flat store can't express.

This demo uses Strands Agents. The patterns are framework-agnostic and carry over to other agent frameworks.

## Prerequisites

1. `OPENAI_API_KEY` (used by the model, gpt-4o-mini)
2. For the graph tests: a running Neo4j (Desktop, Docker, or Aura) with `NEO4J_*` values in `.env`

```bash
uv venv && uv pip install -r requirements.txt
cp .env.example .env
```

## Install dependencies

Run this once (or install from a terminal with `uv venv && uv pip install -r requirements.txt`).

In [ ]:
%pip install -q -r requirements.txt

## Configure your model provider

The demo runs with **OpenAI** by default, but you can use **Amazon Bedrock**, **Anthropic**, or any provider available in the Strands configuration; see [supported model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

- **OpenAI (default):** set `OPENAI_API_KEY` below or in a `.env` file. Get one at https://platform.openai.com/api-keys
- **Amazon Bedrock:** no OpenAI key needed: uses your AWS credentials (`aws configure`, with model access enabled in your region). In the model cell below, comment the `OpenAIModel` lines and uncomment the Bedrock block.

In [ ]:
import os

# python-dotenv loads OPENAI_API_KEY (and any other config) from a local .env file,
# so you don't have to export variables in every new terminal or notebook kernel.
from dotenv import load_dotenv
load_dotenv()

# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Or uncomment and set your key here (not needed for Bedrock)

USING_OPENAI = True  # set False if you switch to Bedrock in the model cell below
if USING_OPENAI:
    assert os.getenv('OPENAI_API_KEY'), (
        'OPENAI_API_KEY not set. '
        'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file. '
        'or switch to Amazon Bedrock in the model cell below.'
    )
print('Provider configured')

## Create the model

Just the engine: one model object, reused by every agent in this notebook.

In [ ]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')  # api_key read from the OPENAI_API_KEY env var

# To run on Amazon Bedrock instead (no OpenAI key; uses your AWS credentials),
# comment the two lines above and uncomment these two:
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

print('Model ready')

---
## Test 1: The problem, the reasoning is lost across a restart

The agent decides in one session. Then we restart it with Strands' own session
management: a new `Agent` instance restores the SAME session from storage (not a
hand-made copy). Asked WHY after the restart, it has no recorded reasoning to
consult, so the answer is a plausible reconstruction. The session carries the
conversation across the restart, but never the tool-by-tool reasoning: that is the
gap the recorder in Test 2 fills.


In [ ]:
# Agent is the Strands agent loop: it calls the model, runs tools, and loops until done.
# SnapshotSessionManager persists the session (messages + agent.state) so a restart
# restores it, no hand-made 'later agent'.
from strands import Agent
from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage
import tempfile

import trace_kv as kv

SYSTEM_PROMPT = 'You are a travel assistant. Use your tools to answer. Be concise: 2-3 sentences maximum.'
SESSION_DIR = tempfile.mkdtemp(prefix='reasoning-sessions-')

def session_manager(session_id):
    # Same session_id + storage => a real restart restores the same conversation and state.
    return SnapshotSessionManager(session_id=session_id, storage=LocalFileStorage(SESSION_DIR))

# Session 1: the agent decides. No recorder attached, so nothing records the reasoning.
deciding_agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                       tools=[kv.search_flights, kv.check_fare_alert],
                       session_manager=session_manager('no-trace-demo'), callback_handler=None)
decision = deciding_agent('Find me a flight JFK to Madrid on 2026-10-10 and pick the best option.')
print('Decision made:', str(decision).strip())


In [ ]:
# Restart: a brand-new Agent instance restores the SAME session from storage.
restarted = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                  tools=[kv.search_flights, kv.check_fare_alert],
                  session_manager=session_manager('no-trace-demo'), callback_handler=None)
answer = restarted('Why did you recommend that Madrid flight?')
print('After restart, asked WHY:', str(answer).strip())

# No recorder ran, so no trace was ever persisted: nothing real is recoverable.
steps_recoverable = len(restarted.state.get(kv.TRACES_KEY) or [])
print('\nReal reasoning steps recoverable from the store:', steps_recoverable)
print('Whatever the answer says, it is a plausible reconstruction, not the real chain.')


---
## Test 2: The recorder captures the trace, and it survives a restart

Same tools, same question, one change: `Agent(hooks=[DecisionTraceRecorder()])`. The
recorder writes the trace into `agent.state`, which the session manager persists. After
a restart (a new `Agent` instance restoring the same session), the trace is still there
and `why_did_i` replays the REAL chain, so the reasoning survives across sessions.


### The recorder, in full

This is the whole mechanism: a `HookProvider` that subscribes to three lifecycle
events the agent already emits. It opens a trace on `BeforeInvocationEvent`, appends
one step per `AfterToolCallEvent` (tool, input, evidence), and persists the finished
trace to `agent.state` on `AfterInvocationEvent`. The tools are never modified; the
recorder reads what they already emit. It is defined here rather than hidden, since
it is the point of the demo. `trace_kv` keeps an identical copy for the `.py` script.

In [ ]:
from strands.hooks import (
    BeforeInvocationEvent, AfterToolCallEvent, AfterInvocationEvent,
    HookProvider, HookRegistry,
)

TRACES_KEY = "decision_traces"


class DecisionTraceRecorder(HookProvider):
    """Records one decision trace per agent invocation into agent.state.

    Attach with Agent(hooks=[DecisionTraceRecorder()]). No tool changes: it reads
    the lifecycle events every tool call already emits.
    """

    def __init__(self):
        self._current = None

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeInvocationEvent, self._on_start)
        registry.add_callback(AfterToolCallEvent, self._on_tool)
        registry.add_callback(AfterInvocationEvent, self._on_end)

    def _on_start(self, event: BeforeInvocationEvent) -> None:
        # Open a trace with the user's question.
        self._current = {'question': kv._last_user_text(event.messages), 'steps': []}

    def _on_tool(self, event: AfterToolCallEvent) -> None:
        # One step per tool call: the tool, its input, and the evidence it produced.
        if self._current is None:
            return
        self._current['steps'].append({
            'n': len(self._current['steps']) + 1,
            'tool': event.tool_use['name'],
            'input': event.tool_use.get('input') or {},
            'evidence': {'name': f"{event.tool_use['name']}-result",
                         'text': kv._result_text(event.result),
                         'source': event.tool_use['name']},
        })

    def _on_end(self, event: AfterInvocationEvent) -> None:
        # Close the trace with the outcome and persist it to agent.state.
        if self._current is None:
            return
        traces = event.agent.state.get(TRACES_KEY) or []
        self._current['id'] = f'trace-{len(traces) + 1}'
        self._current['outcome'] = str(event.result).strip()
        traces.append(self._current)
        event.agent.state.set(TRACES_KEY, traces)
        self._current = None


print('DecisionTraceRecorder defined (a HookProvider, no tool changes)')

In [ ]:
# DecisionTraceRecorder is a Strands HookProvider: it subscribes to the lifecycle
# events every tool call already emits, so it records traces with ZERO tool changes.
# The trace lands in agent.state, which the session manager persists across restarts.
agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
              tools=[kv.search_flights, kv.check_fare_alert, kv.why_did_i],
              hooks=[kv.DecisionTraceRecorder()],
              session_manager=session_manager('recorder-demo'), callback_handler=None)
decision = agent('Find me a flight JFK to Madrid on 2026-10-10 and pick the best option.')
print('Decision made:', str(decision).strip())

traces = agent.state.get(kv.TRACES_KEY) or []
print(f'\nRecorded automatically: {len(traces)} trace(s), {sum(len(t["steps"]) for t in traces)} step(s)')
for step in traces[0]['steps']:
    print(f"  step {step['n']}: {step['tool']}({step['input']}) -> {step['evidence']['text']}")


In [ ]:
# Restart: a new Agent instance restores the same session, trace and all.
restarted = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                  tools=[kv.search_flights, kv.check_fare_alert, kv.why_did_i],
                  hooks=[kv.DecisionTraceRecorder()],
                  session_manager=session_manager('recorder-demo'), callback_handler=None)
answer = restarted('Why did you recommend that Madrid flight?')
print('After restart, asked WHY (agent replays its own trace):', str(answer).strip())

trace = kv.replay_why(restarted.state.get(kv.TRACES_KEY) or [], 'Madrid')
print(f'\nReal reasoning steps recoverable after restart: {len(trace["steps"]) if trace else 0}')


---
## Test 3: The graph track, traces as node chains with provenance edges

The same traces stored in Neo4j:

```
(:Decision)-[:HAS_STEP]->(:Step)-[:NEXT]->(:Step)     the reasoning chain
(:Step)-[:USED]->(:Evidence)                           what each step relied on
(:Evidence)-[:DERIVED_FROM]->(:Evidence)               evidence built on other evidence
(:Evidence)-[:FROM_SOURCE]->(:Source)                  external origin
```

The demo seeds a known 5-decision history (deterministic, reproducible): two flight choices, a budget built **on those choices' outputs**, an itinerary built **on the budget**, and a weather-based packing list as the control.

In [ ]:
# trace_graph is the Neo4j track: the same traces stored as node chains with
# provenance edges, in this demo's own isolated database (reasoningdemo).
import trace_graph as tg

driver = tg.get_driver()
db = tg.ensure_database(driver)
tg.reset_graph(driver, db)
summary = tg.seed_graph(driver, db)
print(f"Seeded: {summary['decisions']} decisions, {summary['evidence']} evidence records, {summary['sources']} external sources")

In [ ]:
# Replay works the same as the flat store: one traversal instead of one lookup.
trace = tg.replay_why_graph(driver, db, "Iberia")
print("Replay 'why the Madrid flight?':", trace["question"])
for step in trace["steps"]:
    print(f"  step {step['n']}: {step['tool']}({step['input']}) -> {step['evidence']}")
print("Outcome:", trace["outcome"])

---
## Test 4: The reverse audit. "this source was wrong, which decisions relied on it?"

The fare-alerts feed is declared compromised. Ground truth by construction: **6 of 8**
decisions depend on it: 2 directly, 4 only through other decisions' outputs, at depths
of one, two, three, and four hops. Two controls depend on other sources.

- The **flat scan** checks each trace's own blob, finds only direct citations.
- The **graph traversal** follows `DERIVED_FROM*0..`, finds them all at any depth, with the evidence path as a receipt.


In [ ]:
ground_truth = sorted(kv.AFFECTED_IDS)
print(f"Ground truth: {len(ground_truth)} affected decisions {ground_truth}\n")

kv_found = sorted(kv.find_affected_decisions_kv(kv.SEED_TRACES, kv.COMPROMISED_SOURCE))
print(f"Flat scan (key-value): found {len(kv_found)}/{len(ground_truth)}  {kv_found}")
missed = sorted(set(ground_truth) - set(kv_found))
print(f"Missed (indirect deps): {missed}\n")

graph_found = sorted(tg.find_affected_decisions_graph(driver, db))
print(f"Graph traversal: found {len(graph_found)}/{len(ground_truth)}  {graph_found}")
controls_clean = all(c not in kv_found and c not in graph_found for c in kv.CONTROL_IDS)
print(f"Controls {sorted(kv.CONTROL_IDS)} correctly NOT flagged by either:", controls_clean)


In [ ]:
# The receipts: the exact evidence path from each indirect decision to the source.
for decision_id in missed:
    hops = tg.provenance_path(driver, db, decision_id)
    print(f"{decision_id}: {' -> '.join(hops)}")


---
### Deterministic vs model-based here

The control lives in the agent's harness: the `DecisionTraceRecorder` is a Strands
[`HookProvider`](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)
attached with `Agent(hooks=[...])`, not a wrapper around the agent. And the audit
track itself is fully deterministic code: the recorder assembling a trace from
lifecycle events, the provenance graph, the `DERIVED_FROM*0..` traversal, and the
flat scan all return the same result for the same input, which is why the 2/6 vs
6/6 scorecard reproduces with no LLM judge.

The one model-based part is upstream: the agent choosing which tools to call as it
makes each decision. A model call carries no reproducibility guarantee, neural-network
inference on GPUs varies with floating-point non-associativity and batching even under
greedy decoding ([Enabling Determinism in LLM Inference](https://arxiv.org/abs/2601.17768),
2026). Recording, storing, and auditing that decision afterwards is deterministic; the
deciding was not. That split is the point: an audit trail must be reproducible even when
the thing it audits is not.

---
## Test 5: Why store the reasoning at all? Tokens saved, errors avoided

This is the payoff. Answering "why did you decide X?" two ways:

1. **Read the stored trace**: no model call, so zero tokens, and the answer is the
   real recorded chain (deterministic).
2. **Ask the model to reconstruct it** with no trace: it costs tokens *and* the answer
   is confabulated, because the real chain was never kept.

So storing the trace **saves tokens** (no model call to replay a past decision) and
**avoids errors** (the real chain instead of a plausible guess). That is the reason
the recorder earns its keep, on top of the reverse audit above.


In [ ]:
# (a) Replay from the stored trace: pure lookup, no model call.
trace = kv.replay_why(kv.SEED_TRACES, 'Madrid')
replay_tokens = 0
print(f'(a) Replay from the stored trace: {replay_tokens} model tokens, '
      f'{len(trace["steps"])} real steps (deterministic).')

# (b) Ask the model to reconstruct the reasoning with no trace to consult.
reconstructor = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT, callback_handler=None)
resp = reconstructor('Earlier you recommended the Iberia JFK-MAD flight. '
                     'Reconstruct the exact tool-by-tool reasoning chain you used.')
reconstruct_tokens = resp.metrics.accumulated_usage['totalTokens']
print(f'(b) Reconstruct with the model: {reconstruct_tokens} model tokens, '
      f'and the chain is confabulated (no trace existed).')

print(f'\nTokens saved per replay: {reconstruct_tokens} -> 0. '
      f'Errors avoided: the real chain vs a plausible guess.')


---
## Summary

| Store | Replay "why did I decide X?" | Reverse audit "source S was wrong" |
|-------|------------------------------|-------------------------------------|
| **Key-value** (flat scan) | one lookup | **2/4** (direct citations only) |
| **Graph** (traversal) | one traversal | **4/4** (any depth, with receipts) |

**Key insight:** both stores replay individual decisions equally well. The graph earns its keep on the *reverse* question: a flat blob never mentions sources it depends on indirectly, while the provenance traversal follows evidence through other decisions' outputs.

![Reverse audit](images/reasoning-memory-reverse-audit.png)

**Scope notes:** the recorder captures tool calls and outcomes, not the model's internal chain-of-thought. "Reasoning memory" is an engineering pattern; the research-backed theme is traceability/provenance (MemWeaver, Engram). All numbers are deterministic checks against the seeded history: no LLM judge.

---
## Cleanup (optional): full teardown

The graph track created an isolated Neo4j database (`reasoningdemo`) with the decision/evidence/source nodes and provenance edges. The cell below removes **all** of it with `teardown_graph`: it clears the demo's nodes and then drops the database entirely. It is guarded, so on Neo4j Community (where the demo falls back to the default `neo4j` database) it only clears this demo's nodes and never drops the shared default. The key-value track holds nothing to clean (it lives in memory).


In [ ]:
# Full teardown of the graph track: clears the demo's nodes, then DROPs the
# isolated database (never the shared default). Safe to run repeatedly.
import trace_graph as tg
try:
    driver
except NameError:
    driver = tg.get_driver()
    db = tg.ensure_database(driver)

tg.teardown_graph(driver, db)
driver.close()
print('Teardown complete: nothing this demo created is left in Neo4j.')
